# docflow

This is a short introduction to docflow.

#### Imports
The public API is exposed through the `docflow` module.

In [ ]:
import docflow as df

#### Datasets

The letter classification dataset is a small dataset of letters and corresponding labels:

```text
letters/
├── A.pdf
├── B.pdf
├── C.pdf
└── labels.csv
```

Labels are stored in a CSV file with the columns `document_id` representing the document filename and `label`.

```csv
document_id,label
A,A
B,B
C,C
```

A `Dataset` groups documents and optional true labels.
- `dataset.documents` contains parsed `Document` objects.
- `dataset.labels` contains true labels loaded from `labels.csv`, or `None` if no label file exists.

In [ ]:
dataset = df.dataset_from_folder("../data/letters")

print(dataset)
print(f"Number of documents: {len(dataset.documents)}")
print(f"Labels: {dataset.labels}")

Each `Document` has an `id`, the `path` on disk, and the extracted `text` content.

In [ ]:
for document in dataset.documents:
    print(f"Document ID: {document.id}")
    print(f"Path: {document.path}")
    print(f"Text: {document.text[:100]}...")  # Print first 100 characters of text
    print()

#### LLM backend

A backend is responsible for talking to an LLM. The current implementation uses [LiteLLM](https://github.com/BerriAI/litellm). We use a local Ollama server, which can be started with:

```bash
ollama serve
```

In [ ]:
backend = df.LiteLLMBackend(model="ollama/mistral:latest", url="http://localhost:11434")

backend

For quick tests without a backend, you can use the `DummyBackend`, which will return a fixed response for any prompt.

#### Prompt templates

`PromptTemplate` supports two rendering stages:
- `.partial(...)` replaces placeholders immediately.
- `.render(...)` replaces placeholders later, for example once a document is known.

Here, `{labels}` is replaced immediately, while `{document_text}` is replaced for each document.

In [ ]:
allowed_labels = ["A", "B", "C"]

system_prompt = df.PromptTemplate("""
You are a strict document classifier.
Return only valid JSON with the keys "label" and "rationale".
""")

user_prompt = df.PromptTemplate("""
Classify the following document.

Allowed labels:
{labels}
                                
Document:
{document_text}
""").partial(labels=allowed_labels)

Inspect the partially rendered user prompt. The `{labels}` placeholder has been replaced, while `{document_text}` remains for per-document rendering.

In [ ]:
print(user_prompt.text)

#### Generate raw responses

`generate_responses` sends each document to the backend and returns `Response` objects. This layer is general. It does not know whether the task is classification, summarization, extraction, or something else.

In [ ]:
responses = df.generate_responses(
    backend=backend,
    documents=dataset.documents,
    user_prompt=user_prompt,
    system_prompt=system_prompt,
)

for response in responses:
    print(f"Document ID: {response.document_id}")
    print(f"Response Text: {response.text}")
    print()

#### Parse responses into classifications

`classifications_from_responses` parses the response text as JSON and checks that the returned label is one of the allowed labels.

In [ ]:
classifications = df.classifications_from_responses(
    responses=responses, labels=allowed_labels
)

for classification in classifications:
    print(f"Document ID: {classification.document_id}")
    print(f"Predicted Label: {classification.label}")
    print()

There is also a convenience function that combines response generation and classification parsing:

In [ ]:
classifications = df.classify_documents(
    documents=dataset.documents,
    backend=backend,
    labels=allowed_labels,
    user_prompt=user_prompt,
    system_prompt=system_prompt,
)

for classification in classifications:
    print(f"Document ID: {classification.document_id}")
    print(f"Predicted Label: {classification.label}")
    print()

#### Evaluation

If the dataset contains true labels, we can compute and plot a confusion matrix.

In [ ]:
if dataset.labels is None:
    raise ValueError("Dataset does not contain labels.")

matrix, matrix_labels = df.confusion_matrix(
    classifications=classifications,
    true_labels=dataset.labels,
)

df.plot_confusion_matrix(
    matrix=matrix,
    labels=matrix_labels,
    output_path=None,
)

#### Summary

`docflow` has these core concepts:
- `Document`: a parsed input document
- `Dataset`: a collection of documents and optional true labels
- `PromptTemplate`: a template for user and system prompts
- `LLMBackend`: an interface to an LLM
- `Response`: a raw response from the LLM
- `Classification`: a parsed classification result

The main workflow is:

```python
dataset = df.dataset_from_folder(...)
user_prompt = df.PromptTemplate(...)
system_prompt = df.PromptTemplate(...)

responses = df.generate_responses(
    dataset=dataset,
    user_prompt=user_prompt,
    system_prompt=system_prompt
)

classifications = df.classifications_from_responses(
    responses=responses,
    allowed_labels=allowed_labels
)
```